# DefectosCafeVerde — final DVSR1 locked test

Final reporting only. Extracts only grouped test, performs no fitting, and ends the study regardless of outcome.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, importlib, json, os, shutil, subprocess, sys, tarfile
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'; BRANCH='codex/defectoscafeverde-dual-view-fusion'
ARCHIVE_NAME='defectoscafeverde-grouped-physical-v1.tar'; EXPECTED_BYTES=686474752; EXPECTED_SHA='53fb2233f1f0d1c77cb24eca2d720f86e0a16835b8a69f4e8f3176fae1aacef2'
def sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda:stream.read(8*1024*1024),b''): digest.update(chunk)
    return digest.hexdigest()
roots=[Path('/content/drive/MyDrive'),Path('/content/drive/.shortcut-targets-by-id')]
result_name='DVSR1_validation_result.json'
matches=sorted({p for root in roots if root.exists() for p in root.rglob(result_name) if p.is_file() and p.parent.name=='defectoscafeverde-dvsr1-v1'})
if not matches: raise FileNotFoundError(result_name)
by_hash={}
for path in matches: by_hash.setdefault(sha256(path),[]).append(path)
if len(by_hash)!=1: raise RuntimeError(f'Ditemukan DVSR1 result berbeda: {by_hash}')
aliases=next(iter(by_hash.values())); preferred=[p for p in aliases if '/MyDrive/' in p.as_posix()]
DVSR1_RESULT=(preferred or aliases)[0]; PROJECT=DVSR1_RESULT.parents[2]
dv=json.loads(DVSR1_RESULT.read_text()); assert dv['decision']=='PASS' and dv['next']=='PROMOTE_DVSR1_SIDE_RELIABILITY' and dv['test_images_accessed'] is False
drive_roots=[p for p in [Path('/content/drive/MyDrive'),*Path('/content/drive/.shortcut-targets-by-id').glob('*')] if p.exists()]
archives=sorted({p for root in drive_roots for p in root.rglob(ARCHIVE_NAME) if p.is_file()})
valid=[p for p in archives if p.stat().st_size==EXPECTED_BYTES and sha256(p)==EXPECTED_SHA]
if valid: ARCHIVE=valid[0]
else:
    complete=[]
    for scheme,count in (('part16',41),('chunk',11),('part',2)):
        by_parent={}
        for root in drive_roots:
            for path in root.rglob(f'{ARCHIVE_NAME}.{scheme}-*'):
                if path.is_file(): by_parent.setdefault(path.parent,{})[path.name]=path
        expected=[f'{ARCHIVE_NAME}.{scheme}-{index:03d}' for index in range(count)]
        for parent,rows in by_parent.items():
            if sorted(rows)==expected and sum(rows[name].stat().st_size for name in expected)==EXPECTED_BYTES: complete.append([rows[name] for name in expected])
    if not complete: raise FileNotFoundError(f'Tidak menemukan {ARCHIVE_NAME} atau chunk lengkap')
    ARCHIVE=WORK/ARCHIVE_NAME
    with ARCHIVE.open('wb') as target:
        for part in complete[0]:
            with part.open('rb') as source: shutil.copyfileobj(source,target,length=8*1024*1024)
    if ARCHIVE.stat().st_size!=EXPECTED_BYTES or sha256(ARCHIVE)!=EXPECTED_SHA: raise RuntimeError('Archive tidak valid')
print('PROJECT:',PROJECT); print('ARCHIVE:',ARCHIVE)


In [ ]:
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan GPU T4')
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()); print('GPU:',torch.cuda.get_device_name(0))
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_defectoscafeverde_dvsr1_locked_test.py'],cwd=REPO,check=True)


In [ ]:
DATA=WORK/'defectoscafeverde-locked-test-v1'
if DATA.exists(): shutil.rmtree(DATA)
DATA.mkdir(parents=True); prefix='grouped-physical-v1/'; allowed={'data.yaml','grouped_audit.json','grouped_manifest.json','grouped_summary.json'}
with tarfile.open(ARCHIVE,'r') as bundle:
    for member in bundle.getmembers():
        name=member.name.replace('\\','/');
        if not name.startswith(prefix): continue
        relative=name[len(prefix):].strip('/')
        if not relative: continue
        first=relative.split('/',1)[0]
        if not (first=='test' or relative in allowed): continue
        target=(DATA/relative).resolve()
        if DATA.resolve() not in target.parents and target!=DATA.resolve(): raise RuntimeError('Unsafe archive path')
        if member.isdir(): target.mkdir(parents=True,exist_ok=True); continue
        target.parent.mkdir(parents=True,exist_ok=True); source=bundle.extractfile(member)
        if source is None: raise RuntimeError(member.name)
        with source, target.open('wb') as out: shutil.copyfileobj(source,out)
if (DATA/'train').exists() or (DATA/'val').exists(): raise RuntimeError('Development split terexpose')
for required in (DATA/'test/images',DATA/'test/labels',DATA/'grouped_manifest.json',DATA/'grouped_audit.json'):
    if not required.exists(): raise FileNotFoundError(required)
print('TEST IMAGES:',len(list((DATA/'test/images').glob('*'))),'TRAIN:',(DATA/'train').exists(),'VAL:',(DATA/'val').exists())


In [ ]:
D0_RESULT=PROJECT/'experiments/defectoscafeverde-cwcf-direct-v1/val_reports/D0DIRECT_seed42_result.json'
D0_CHECKPOINT=PROJECT/'experiments/defectoscafeverde-cwcf-direct-v1/D0DIRECT/D0DIRECT_seed42/weights/best.pt'
SEED42=PROJECT/'experiments/defectoscafeverde-dvf1-v1/DVF1_seed42_result.json'
CONFIRM=PROJECT/'experiments/defectoscafeverde-dvf1-confirmation-v1'; SEED123=CONFIRM/'DVF1_seed123_result.json'; SEED2026=CONFIRM/'DVF1_seed2026_result.json'; DECISION=CONFIRM/'DVF1_three_seed_confirmation.json'
TRAIN_AUDIT=PROJECT/'experiments/defectoscafeverde-dvsr-routability-v1/dvsr_routability.json'
OUT=PROJECT/'experiments/defectoscafeverde-dvsr1-locked-test-v1'
for path in (D0_RESULT,D0_CHECKPOINT,SEED42,SEED123,SEED2026,DECISION,TRAIN_AUDIT,DVSR1_RESULT):
    if not path.is_file(): raise FileNotFoundError(path)
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_defectoscafeverde_dvsr1_locked_test','--data-root',str(DATA),'--grouped-manifest',str(DATA/'grouped_manifest.json'),'--grouped-audit',str(DATA/'grouped_audit.json'),'--d0-result',str(D0_RESULT),'--d0-checkpoint',str(D0_CHECKPOINT),'--seed42-result',str(SEED42),'--seed123-result',str(SEED123),'--seed2026-result',str(SEED2026),'--confirmation-decision',str(DECISION),'--train-audit',str(TRAIN_AUDIT),'--dvsr1-result',str(DVSR1_RESULT),'--output-root',str(OUT),'--device','0','--authorize-test']
print('MENJALANKAN FINAL LOCKED TEST — TIDAK ADA TRAINING')
subprocess.run(command,cwd=REPO,check=True)
RESULT=OUT/'DVSR1_locked_test_result.json'; result=json.loads(RESULT.read_text())
assert result['next']=='REPORT_LOCKED_TEST_AND_STOP' and result['test_images_accessed'] is True


In [ ]:
import pandas as pd
metrics=('physical_pair_accuracy','macro_class_accuracy','bottom3_class_accuracy','worst_class_accuracy')
rows=[]
for model in ('PAPER_MAX_CONFIDENCE','DVSR1','DVF1_MEAN','PAIR_ORACLE'):
    rows.append({'model':model,**{metric:result['values'][model][metric] for metric in metrics}})
display(pd.DataFrame(rows).style.format({metric:'{:.2%}' for metric in metrics}))
paper=result['values']['PAPER_MAX_CONFIDENCE']['accuracy_by_class']; candidate=result['values']['DVSR1']['accuracy_by_class']
class_rows=[{'class_name':name,'paper':paper[name],'DVSR1':candidate[name],'delta':candidate[name]-paper[name]} for name in paper]
display(pd.DataFrame(class_rows).sort_values('delta',ascending=False).style.format({'paper':'{:.2%}','DVSR1':'{:.2%}','delta':'{:+.2%}'}))
print('DVF1 BY SEED:',{seed:{metric:row[metric] for metric in metrics} for seed,row in result['values']['DVF1_BY_SEED'].items()})
print('VS PAPER:',result['deltas_vs_paper']); print('VS DVF1 MEAN:',result['deltas_vs_dvf1_mean'])
print('TRANSITIONS:',result['transitions_vs_paper']); print('PAIR STATS:',result['test_pair_stats'])
print('CRITERIA:',result['criteria']); print('GATES:',result['gates'])
print('DECISION:',result['decision'],'| NEXT:',result['next'])
print('TRAINING:',result['detector_training_executed'] or result['fuser_training_executed'] or result['selector_fitting_executed'],'| TEST:',result['test_images_accessed'])
print('SUMMARY:',RESULT)
